# 🚀 RAG2 - نظام تدريب النماذج على Google Colab

هذا Notebook يساعدك على تدريب نموذج 1.25B على Google Colab مع GPU مجاني!

## التعليمات:
1. اضغط على `Runtime` → `Change runtime type`
2. اختر `T4 GPU` أو `A100 GPU`
3. شغل الخلايا بالترتيب

In [ ]:
# 1. تثبيت المكتبات المطلوبة
!pip install torch torchvision transformers datasets accelerate tqdm rich python-dotenv safetensors sentencepiece protobuf wandb

In [ ]:
# 2. تحميل الكود من GitHub (أو ارفع المجلد يدوياً)
!git clone https://github.com/your-username/RAG2.git
%cd RAG2

## أو ارفع المجلد يدوياً:
1. اضغط على أيقونة المجلد في الجانب الأيسر
2. اضغط على أيقونة الرفع
3. ارفع مجلد `RAG2` كامل

In [ ]:
# 3. التحقق من GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 4. استيراد المكتبات
import os
import sys
sys.path.append('/content/RAG2')

from config import MODEL_SIZES, get_model_config, calculate_parameters, VOCAB_SIZE
from model import TransformerModel
from trainer import ModelTrainer, TextDataset, SimpleTokenizer
from rich.console import Console
from rich.progress import Progress
import torch

console = Console()

In [ ]:
# 5. إنشاء بيانات تدريب تجريبية
os.makedirs('data', exist_ok=True)

# نص PHP
php_text = """<?php
$name = "Ahmed";
echo "Hello " . $name;
function add($a, $b) {
    return $a + $b;
}
class Person {
    public $name;
    public function __construct($name) {
        $this->name = $name;
    }
}
""" * 50

# نص HTML
html_text = """<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <title>مرحبا</title>
</head>
<body>
    <h1>مرحبا بالعالم</h1>
    <form action="submit.php" method="POST">
        <input type="text" name="name">
        <button type="submit">إرسال</button>
    </form>
</body>
</html>
""" * 50

# نص عربي
arabic_text = """الذكاء الاصطناعي هو فرع من علوم الحاسوب.
التعلم الآلي يسمح للحاسبات بالتعلم من البيانات.
التعلم العميق يستخدم الشبكات العصبية العميقة.
مرحبا بك في عالم البرمجة.
البرمجة مهارة مستقبلية مهمة.
""" * 50

# حفظ البيانات
with open('data/training_data.txt', 'w', encoding='utf-8') as f:
    f.write(php_text + html_text + arabic_text)

print("✅ تم إنشاء بيانات التدريب")

In [ ]:
# 6. إعداد النموذج
# اختر حجم النموذج: small, medium, large, xl, 1b, 1.25b
MODEL_SIZE = "1.25b"  # غير هذا لحجم آخر إذا أردت

config = get_model_config(MODEL_SIZE)
config['batch_size'] = 4  # GPU Colab لا يتحمل أكثر من 4
config['learning_rate'] = 5e-5

console.print(f"[bold green]Creating model: {MODEL_SIZE}[/bold green]")
model = TransformerModel(config)

params = model.count_parameters()
console.print(f"[cyan]Model parameters: {params:,} ({params/1e9:.2f}B)[/cyan]")
console.print(f"[cyan]Model memory: {params * 4 / 1e9:.2f} GB[/cyan]")

In [ ]:
# 7. تحميل البيانات
tokenizer = SimpleTokenizer(config.get('vocab_size', 50257))
dataset = TextDataset('data', tokenizer, max_length=config.get('max_position_embeddings', 1024))

console.print(f"[green]✓ Loaded {len(dataset)} training samples[/green]")

if len(dataset) == 0:
    console.print("[red]No training data found![/red]")
else:
    console.print(f"[green]First sample length: {len(dataset[0]['input_ids'])} tokens[/green]")

In [ ]:
# 8. بدء التدريب
console.print("[bold cyan]Starting training...[/bold cyan]")

trainer = ModelTrainer(model, config)

# تدريب 1 epoch فقط (يمكنك زيادة العدد)
history = trainer.train(dataset, num_epochs=1)

console.print("[bold green]✅ Training completed![/bold green]")
console.print(f"[cyan]Final loss: {history[-1]['loss'] if history else 0:.4f}[/cyan]")
console.print(f"[cyan]Total steps: {trainer.global_step}[/cyan]")

In [ ]:
# 9. اختبار النموذج
console.print("[bold cyan]Testing model...[/bold cyan]")

test_prompt = "اكتب كود PHP"
input_ids = tokenizer.encode(test_prompt)
input_tensor = torch.tensor([input_ids])

# توليد النص
model.eval()
with torch.no_grad():
    output = model.generate(input_tensor, max_length=50, temperature=0.8)

generated_text = tokenizer.decode(output[0].tolist())
console.print(f"[green]Generated: {generated_text}[/green]")

In [ ]:
# 10. حفظ النموذج
import shutil

# حفظ checkpoint
trainer.save_checkpoint(0, is_best=True)

# ضغط المجلد للتحميل
shutil.make_archive('RAG2_model', 'zip', 'checkpoints')

console.print("[bold green]✅ Model saved![/bold green]")
console.print("[cyan]Download 'RAG2_model.zip' from the file browser on the left[/cyan]")

In [ ]:
# 11. تحميل النموذج (اختياري)
from google.colab import files

try:
    files.download('RAG2_model.zip')
    console.print("[bold green]✅ Download started![/bold green]")
except:
    console.print("[yellow]Download manually from file browser[/yellow]")

## 🎉 تم الانتهاء!

### ملخص:
- ✅ تم تثبيت المكتبات
- ✅ تم إنشاء النموذج
- ✅ تم التدريب
- ✅ تم الاختبار
- ✅ تم الحفظ

### الخطوات التالية:
1. حمّل `RAG2_model.zip`
2. استخدم النموذج في مشروعك

### نصائح:
- استخدم `T4 GPU` أو `A100` للسرعة
- قلل `batch_size` إذا واجهت مشاكل ذاكرة
- زد `num_epochs` للتدريب الأطول